
Installare 

Tesseract

OpenCV

python -m pip install ot
py -m pip install opencv-contrib-python

In [ ]:

import os
import cv2
import numpy as np
import pytesseract
from pdf2image import convert_from_path
import json

PDF_FOLDER= f"data/pdf/"
TXT_FOLDER= f"data/txt/"


In [ ]:
def process_pdfs_in_folder(pdf_folder, txt_folder, tesseract_cmd):
    # esegue Tesseract
    pytesseract.pytesseract.tesseract_cmd = tesseract_cmd    
    # Scansiona i file nella cartella
    for filename in os.listdir(pdf_folder):
        if filename.lower().endswith(".pdf"):
            pdf_path_filename = os.path.join(pdf_folder, filename)
            print(f"esecuzione: {pdf_path_filename}")            
            # Converti il PDF in immagini
            images = convert_from_path(pdf_path_filename)            
            # Estrai il testo da ogni immagine
            extracted_text = ""
            for i, image in enumerate(images):
                text = pytesseract.image_to_string(image)
                extracted_text += text + "\n"
                print(f"pagina {i+1} ...")            
            # Salva il testo estratto in un file di output
            txt_filename = os.path.splitext(filename)[0] + ".txt"
            output_text_path = os.path.join(txt_folder, txt_filename)
            with open(output_text_path, "w", encoding="utf-8") as text_file:
                text_file.write(extracted_text)
            print(f"Salva il testo estratto nel file: {output_text_path}\n")


 
tesseract_cmd='C:/Program Files/Tesseract-OCR/tesseract.exe'
process_pdfs_in_folder(PDF_FOLDER, TXT_FOLDER, tesseract_cmd)

In [ ]:



def process_pdfs_in_folder(
    pdf_folder,
    txt_folder,
    tesseract_cmd,
    poppler_path,
    skip_existing=True,
    save_json=False
):

    pytesseract.pytesseract.tesseract_cmd = tesseract_cmd

    os.makedirs(txt_folder, exist_ok=True)

    for filename in os.listdir(pdf_folder):

        if not filename.lower().endswith(".pdf"):
            continue

        pdf_path = os.path.join(pdf_folder, filename)
        txt_filename = os.path.splitext(filename)[0] + ".txt"
        txt_path = os.path.join(txt_folder, txt_filename)

        # SKIP opzionale
        if skip_existing and os.path.exists(txt_path):
            print(f"⏭️ skip: {filename}")
            continue

        print(f"\n📄 elaboro: {filename}")

        try:

            images = convert_from_path(
                pdf_path,
                dpi=300,
                poppler_path=poppler_path
            )

            extracted_text = ""
            pages = []

            for i, image in enumerate(images):

                print(f"pagina {i+1}")

                img = cv2.cvtColor(np.array(image), cv2.COLOR_BGR2GRAY)
                img = cv2.threshold(
                    img, 0, 255,
                    cv2.THRESH_BINARY + cv2.THRESH_OTSU
                )[1]

                text = pytesseract.image_to_string(
                    img,
                    lang="ita",
                    config="--psm 6"
                )

                extracted_text += text + "\n"

                if save_json:
                    pages.append({
                        "page": i + 1,
                        "text": text
                    })

            # salva TXT
            with open(txt_path, "w", encoding="utf-8") as f:
                f.write(extracted_text)

            print(f"✅ salvato TXT: {txt_path}")

            # salva JSON opzionale
            if save_json:

                json_filename = os.path.splitext(filename)[0] + ".json"
                json_path = os.path.join(txt_folder, json_filename)

                with open(json_path, "w", encoding="utf-8") as f:
                    json.dump({
                        "file": filename,
                        "pages": pages
                    }, f, indent=2, ensure_ascii=False)

                print(f"✅ salvato JSON: {json_path}")

        except Exception as e:

            print(f"❌ errore su {filename}")
            print(e)

In [ ]:
PDF_FOLDER= r"prova/"
TXT_FOLDER= r"prova/"

TESSERACT = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
POPPLER = r"C:\Program Files\poppler\poppler-25.12.0\Library\bin"

process_pdfs_in_folder(
    PDF_FOLDER,
    TXT_FOLDER,
    TESSERACT,
    POPPLER,
    skip_existing=True,   # skip se TXT già esiste
    save_json=False       # JSON opzionale
)

In [11]:
import re
import csv
from pathlib import Path

REGIONI = [
    "Abruzzo", "Basilicata", "Calabria", "Campania", "Emilia-Romagna",
    "Friuli-Venezia Giulia", "Lazio", "Liguria", "Lombardia", "Marche",
    "Molise", "Piemonte", "Puglia", "Sardegna", "Sicilia", "Toscana",
    "Umbria", "Valle d'Aosta/Vallée d'Aoste", "Veneto", "P.A. Trento",
    "P.A. Bolzano", "Trentino", "Sardergna"
]

PROVINCE_NOTE = [
    "L'Aquila", "Reggio Calabria", "Forlì-Cesena", "Reggio nell'Emilia",
    "Verbano-Cusio-Ossola", "Pesaro e Urbino", "Sud Sardegna",
    "Benevento/Avellino", "Benevento/ Avellino", "Ge", "Im", "Sv", "Tn"
]

def clean_text(txt: str) -> str:
    txt = txt.replace("\ufeff", " ")
    txt = txt.replace("—", "-").replace("–", "-")
    txt = txt.replace("’", "'").replace("“", '"').replace("”", '"')
    txt = txt.replace("|", " ")
    txt = re.sub(r"[ \t]+", " ", txt)
    txt = re.sub(r"\n+", "\n", txt)
    return txt.strip()

def split_records(text: str):
    """
    Divide il testo in blocchi usando il numero iniziale del record.
    """
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

    # scarta intestazioni/totali
    filtered = []
    for ln in lines:
        up = ln.upper()
        if up.startswith("TOT.") or "DENOMINAZIONE DELL'UNIONE" in up:
            continue
        if "TOTALE UNIONI" in up or "TOTALE POPOLAZIONE" in up or "TOTALE COMUNI" in up:
            continue
        filtered.append(ln)

    records = []
    current_num = None
    current_lines = []

    for ln in filtered:
        m = re.match(r"^\s*(\d{1,3})\b", ln)
        if m:
            n = int(m.group(1))
            if 1 <= n <= 396:
                if current_num is not None:
                    records.append((current_num, " ".join(current_lines).strip()))
                current_num = n
                current_lines = [ln]
                continue

        if current_num is not None:
            current_lines.append(ln)

    if current_num is not None and current_lines:
        records.append((current_num, " ".join(current_lines).strip()))

    return records

def find_regione(text: str):
    for reg in sorted(REGIONI, key=len, reverse=True):
        if reg in text:
            return reg
    return ""

def find_provincia(after_regione: str):
    # prima province note multi-token
    for prov in sorted(PROVINCE_NOTE, key=len, reverse=True):
        if after_regione.startswith(prov + " "):
            return prov

    # fallback: fino a prima di una sequenza che sembri comuni o popolazione
    # molto semplice: prende 1-3 token iniziali
    tokens = after_regione.split()
    if not tokens:
        return ""

    # prova 3, 2, 1 token
    for k in (3, 2, 1):
        cand = " ".join(tokens[:k])
        if cand in PROVINCE_NOTE:
            return cand

    # fallback brutale: primo token
    return tokens[0]

def parse_record(n, raw):
    """
    Prova a estrarre:
    n, denominazione, regione, provincia, comuni, popolazione, n_comuni
    """
    s = raw.strip()

    # rimuove il numero iniziale dal testo
    s = re.sub(rf"^\s*{n}\s*", "", s).strip()

    # popolazione e numero comuni in coda
    popolazione = ""
    n_comuni = ""

    m_tail = re.search(r"(\d{1,3}(?:\.\d{3})*)(?:\s+(\d{1,2}))?\s*$", s)
    if m_tail:
        popolazione = m_tail.group(1)
        if m_tail.group(2):
            n_comuni = m_tail.group(2)
        s = s[:m_tail.start()].strip()

    regione = find_regione(s)
    provincia = ""
    denominazione = ""
    comuni = ""

    if regione:
        parts = s.split(regione, 1)
        left = parts[0].strip()
        right = parts[1].strip()

        denominazione = left
        provincia = find_provincia(right)

        if provincia and right.startswith(provincia):
            comuni = right[len(provincia):].strip()
        else:
            comuni = right.strip()
    else:
        # fallback: tutto dentro denominazione/comuni
        denominazione = s

    # pulizie finali
    denominazione = re.sub(r"\s+", " ", denominazione).strip(" ;")
    regione = re.sub(r"\s+", " ", regione).strip(" ;")
    provincia = re.sub(r"\s+", " ", provincia).strip(" ;")
    comuni = re.sub(r"\s+", " ", comuni).strip(" ;,")

    return {
        "n": n,
        "denominazione_unione": denominazione,
        "regione": regione,
        "provincia": provincia,
        "comuni": comuni,
        "popolazione": popolazione,
        "n_comuni": n_comuni,
        "raw": raw
    }

def txt_to_semicolon_csv(input_txt, output_csv):
    text = Path(input_txt).read_text(encoding="utf-8", errors="ignore")
    text = clean_text(text)

    records = split_records(text)
    parsed = [parse_record(n, raw) for n, raw in records]

    with open(output_csv, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f, delimiter=";")
        writer.writerow([
            "n",
            "denominazione_unione",
            "regione",
            "provincia",
            "comuni",
            "popolazione",
            "n_comuni"
        ])
        for r in parsed:
            writer.writerow([
                r["n"],
                r["denominazione_unione"],
                r["regione"],
                r["provincia"],
                r["comuni"],
                r["popolazione"],
                r["n_comuni"]
            ])

    return parsed


# ESEMPIO USO
input_txt = r"prova/Elenco-Unioni-di-Comuni-anno-2023-1.txt"
output_csv = r"prova/unioni_comuni_da_txt.csv"

records = txt_to_semicolon_csv(input_txt, output_csv)
print(f"Record estratti: {len(records)}")
print("Primi 3:")
for r in records:
    print(r)

Record estratti: 333
Primi 3:
{'n': 1, 'denominazione_unione': 'Unione dei Comuni del Sinello', 'regione': 'Abruzzo', 'provincia': 'Chieti', 'comuni': 'Carpineto Sinello,Carunchio,Dogliola,Guilmi Montazzoli,Palmoli,San Giovanni 4.864 Lipioni,Torrebruna, Tufillo', 'popolazione': '', 'n_comuni': '', 'raw': '1 Unione dei Comuni del Sinello Abruzzo Chieti Carpineto Sinello,Carunchio,Dogliola,Guilmi Montazzoli,Palmoli,San Giovanni 4.864 Lipioni,Torrebruna, Tufillo'}
{'n': 2, 'denominazione_unione': 'Unione dei Comuni della Vallata del Foro', 'regione': 'Abruzzo', 'provincia': 'Chieti', 'comuni': '8.803 Ari,Ripa Teatina, Vacri, Villamagna', 'popolazione': '', 'n_comuni': '', 'raw': '2 Unione dei Comuni della Vallata del Foro Abruzzo Chieti 8.803 Ari,Ripa Teatina, Vacri, Villamagna'}
{'n': 3, 'denominazione_unione': 'Unione Dei Comuni Montani Maiella Orientale-Verde', 'regione': 'Abruzzo', 'provincia': 'Chieti', 'comuni': '2.102 Aventino Colledimacine,Lettopalena,Palena,Taranta Peligna', 'pop

In [12]:
import re
import csv
from pathlib import Path

def parse_union_blocks(txt_path, out_csv_path):
    text = Path(txt_path).read_text(encoding="utf-8", errors="ignore")
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

    # rimuovi intestazioni
    clean = []
    for ln in lines:
        up = ln.upper()
        if "DENOMINAZIONE DELL'UNIONE" in up:
            continue
        if up.startswith("TOT.") or "TOTALE UNIONI" in up:
            continue
        clean.append(ln)

    rows = []
    i = 0

    while i < len(clean):
        # inizio record: numero 1..396
        if re.fullmatch(r"\d{1,3}", clean[i]):
            n = int(clean[i])

            # struttura attesa a blocchi
            denominazione = clean[i + 1] if i + 1 < len(clean) else ""
            reg_prov      = clean[i + 2] if i + 2 < len(clean) else ""
            comuni_line   = clean[i + 3] if i + 3 < len(clean) else ""
            tail_line     = clean[i + 4] if i + 4 < len(clean) else ""

            # regione = primo token "noto", provincia = resto
            regione = ""
            provincia = ""

            regioni = [
                "Abruzzo","Basilicata","Calabria","Campania","Emilia-Romagna",
                "Friuli-Venezia Giulia","Lazio","Liguria","Lombardia","Marche",
                "Molise","Piemonte","Puglia","Sardegna","Sicilia","Toscana",
                "Umbria","Veneto","Valle d'Aosta/Vallée d'Aoste",
                "P.A. Trento","P.A. Bolzano","Trentino-Alto Adige"
            ]

            for reg in sorted(regioni, key=len, reverse=True):
                if reg_prov.startswith(reg):
                    regione = reg
                    provincia = reg_prov[len(reg):].strip()
                    break

            # esplodi comuni
            comuni = [c.strip() for c in comuni_line.split(",") if c.strip()]

            for comune in comuni:
                rows.append({
                    "n": n,
                    "unione": denominazione,
                    "regione": regione,
                    "provincia": provincia,
                    "comune": comune
                })

            i += 5
        else:
            i += 1

    with open(out_csv_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["n", "unione", "regione", "provincia", "comune"],
            delimiter=";"
        )
        writer.writeheader()
        writer.writerows(rows)

    return rows


# uso
rows = parse_union_blocks(
    "prova/Elenco-Unioni-di-Comuni-anno-2023-1.txt",
    "unioni_comuni_coppie_blocchi.csv"
)

print("Righe create:", len(rows))
print(rows[:10])

Righe create: 0
[]


In [19]:
import re
import csv
from pathlib import Path

REGIONI = [
    "Abruzzo", "Basilicata", "Calabria", "Campania", "Emilia-Romagna",
    "Friuli-Venezia Giulia", "Lazio", "Liguria", "Lombardia", "Marche",
    "Molise", "Piemonte", "Puglia", "Sardegna", "Sicilia", "Toscana",
    "Umbria", "Veneto", "Valle d'Aosta/Vallée d'Aoste",
    "P.A. Trento", "P.A. Bolzano", "Trentino-Alto Adige"
]

def pulisci_testo(txt: str) -> str:
    txt = txt.replace("\ufeff", " ")
    txt = txt.replace("—", "-").replace("–", "-")
    txt = txt.replace("’", "'").replace("“", '"').replace("”", '"')
    txt = txt.replace("￾", " ")
    txt = re.sub(r"[ \t]+", " ", txt)
    txt = re.sub(r"\n+", "\n", txt)
    return txt.strip()

def separa_regione_provincia(riga: str):
    riga = riga.strip()
    for regione in sorted(REGIONI, key=len, reverse=True):
        if riga.startswith(regione):
            provincia = riga[len(regione):].strip()
            return regione, provincia
    return "", riga

def estrai_blocchi(lines):
    """
    Ogni record inizia con una riga che contiene solo un numero da 1 a 396.
    """
    blocchi = []
    i = 0
    while i < len(lines):
        if re.fullmatch(r"\d{1,3}", lines[i]):
            n = int(lines[i])
            if 1 <= n <= 396:
                blocco = [lines[i]]
                i += 1
                while i < len(lines) and not re.fullmatch(r"\d{1,3}", lines[i]):
                    blocco.append(lines[i])
                    i += 1
                blocchi.append(blocco)
                continue
        i += 1
    return blocchi

def parse_blocco(blocco):
    """
    Struttura attesa:
    [0] numero
    [1] nome unione
    [2] regione + provincia
    [3..n-2] comuni (eventualmente spezzati su più righe)
    [n-1] popolazione + num comuni  -> da ignorare
    """
    if len(blocco) < 4:
        return []

    numero = blocco[0].strip()
    nome_unione = blocco[1].strip()
    regione_provincia = blocco[2].strip()

    regione, provincia = separa_regione_provincia(regione_provincia)

    # Tutte le righe tra regione/provincia e l'ultima riga numerica finale
    righe_comuni = blocco[3:]

    # se l'ultima riga sembra "13.404 7" oppure "294.045 95", la scartiamo
    if righe_comuni and re.fullmatch(r"\d{1,3}(?:\.\d{3})*(?:\s+\d{1,3})?", righe_comuni[-1].strip()):
        righe_comuni = righe_comuni[:-1]

    testo_comuni = " ".join(righe_comuni)
    testo_comuni = re.sub(r"\s+", " ", testo_comuni).strip()

    comuni = [c.strip() for c in testo_comuni.split(",") if c.strip()]

    out = []
    for comune in comuni:
        out.append({
            "numero_unione": numero,
            "nome_unione": nome_unione,
            "comune": comune,
            "provincia": provincia,
            "regione": regione
        })
    return out

def txt_to_csv_normalizzato(input_txt, output_csv):
    text = Path(input_txt).read_text(encoding="utf-8", errors="ignore")
    text = pulisci_testo(text)

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

    # togli intestazioni e totali
    righe_pulite = []
    for ln in lines:
        up = ln.upper()
        if "DENOMINAZIONE DELL'UNIONE" in up:
            continue
        if up.startswith("TOT.") or "TOT. POPOLAZIONE" in up or "TOT. COMUNI" in up:
            continue
        if "TOTALE UNIONI" in up:
            continue
        righe_pulite.append(ln)

    blocchi = estrai_blocchi(righe_pulite)

    rows = []
    for blocco in blocchi:
        rows.extend(parse_blocco(blocco))

    with open(output_csv, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f, delimiter=";")
        writer.writerow(["numero_unione", "nome_unione", "comune", "provincia", "regione"])
        for r in rows:
            writer.writerow([
                r["numero_unione"],
                r["nome_unione"],
                r["comune"],
                r["provincia"],
                r["regione"]
            ])

    return rows

In [20]:
input_txt = r"prova/Elenco-Unioni-di-Comuni-anno-2023-1.txt"
output_csv = r"prova/unioni_comuni_normalizzato.csv"

rows = txt_to_csv_normalizzato(input_txt, output_csv)

print("Righe create:", len(rows))
for r in rows:
    print(r)

Righe create: 0
